# CRIVO — Fase 5: o 17M conversando

**Ideia:** o 17M aprendeu português (fase 2) e a dizer 10 atos a partir de um quadro (fase 3), mas não aprendeu a **conversar**. Aqui ele aprende com conversas simples e densas, no jeito de `IDENTIDADE.md`, e o CRIVO entra só como fonte conferida: fatos do acervo, assunto desconhecido, nomes guardados e cuidado de saúde vão no contexto.

**Entrada:** `<documento> contexto <fim>` + as falas anteriores + `<usuario> fala <fim> <assistente>`. **Saída:** a resposta, terminando em `<fim>`.

**Dados** (`experimentos/fase5_conversa_20261013/`, conferidos por SHA): ~78 mil casos de treino, conversas de 1 a 6 turnos, ruído de digitação, 104 situações escritas à mão, os 5.414 fatos do acervo e nomes inventados. O blind tem assuntos, nomes e formas de fala que o treino nunca viu.

**Regra de ouro:** nenhuma informação sobre o mundo fora dos fatos do contexto; números e nomes próprios só do contexto ou da conversa.

**Treino:** parte dos pesos aprovados da fase 3 (`745a2500…`, em `CRIVO/transformer-17m-fase3/`); ajuste fino do modelo inteiro com replay do texto do pré-treino; o melhor ponto é escolhido pelo dev, entre os que preservam a perda de linguagem (≤ 110% da base).

**Gates no blind (fixados antes do treino):** tudo ok ≥ 85%, usa o fato ≥ 90%, nome exato ≥ 90%, "não sei" quando não há fato ≥ 90%, urgência leva ao 192 ≥ 95%, termina ≥ 95%, sem número inventado ≥ 97%, sem nome inventado ≥ 95%, cada situação ≥ 60%, perda de linguagem ≤ 110%. Mesmo passando: porta com as conversas reais pendente, `ativo_no_chat=false`.

> **Ambiente de execução → Alterar tipo → GPU (T4)** → **Executar tudo**. Leva uns 40–70 min. Saída em `CRIVO/transformer-17m-fase5/`.

In [ ]:
# Célula 0 — GPU, Drive e pesos aprovados da fase 3
import os, sys, json, shutil, subprocess, hashlib, time, math, random, gzip
from pathlib import Path
from datetime import datetime
from collections import defaultdict
import numpy as np
import torch

if not torch.cuda.is_available():
    raise SystemExit("ERRO: GPU não disponível. Ambiente de execução → Alterar tipo de ambiente → GPU.")
print("GPU:", torch.cuda.get_device_name(0), "| PyTorch", torch.__version__, flush=True)
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()

APROVADO = "745a2500615559d822f4fdb4d3d1c72850f64a9eac32a210bd6e5881f1d0b293"
MEU = Path("/content/drive/MyDrive")
candidatas = [MEU / "CRIVO"] + sorted(p for p in MEU.iterdir() if p.is_dir() and p.name != "CRIVO")
CRIVO = next((p for p in candidatas if (p / "transformer-17m-fase3").is_dir()), MEU / "CRIVO")
ORIGEM = next((p.parent for p in sorted((CRIVO / "transformer-17m-fase3").glob("fase3_*/pesos.pt")) if sha256(p) == APROVADO), None)
if ORIGEM is None:
    raise SystemExit("ERRO: não achei os pesos aprovados da fase 3 (sha 745a2500…) em CRIVO/transformer-17m-fase3/")
CACHE = CRIVO / "dados-17m"
for p in (CACHE / "treino.bin", CACHE / "validacao.bin"):
    if not p.is_file():
        raise SystemExit(f"ERRO: não encontrei {p} (texto do pré-treino, usado no replay).")
SAIDA = CRIVO / "transformer-17m-fase5"
SAIDA.mkdir(parents=True, exist_ok=True)
dispositivo = torch.device("cuda")
SEED = 20261013
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("Pasta do CRIVO:", CRIVO, "| pesos da fase 3:", ORIGEM, flush=True)


In [ ]:
# Célula 1 — código e dados congelados da fase 5
REPO = Path("/content/CRIVO")
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/affectionate-carson-22yoe0",
                "https://github.com/HROSONE/CRIVO", str(REPO)], check=True)
COMMIT = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tokenizers>=0.20,<1"], check=True)
EXP = REPO / "experimentos" / "fase5_conversa_20261013"
MANIFESTO = json.loads((EXP / "manifesto.json").read_text(encoding="utf-8"))
if sha256(EXP / "dados_fase5.json.gz") != MANIFESTO["arquivo_gz_sha256"]:
    raise SystemExit("ERRO: dados_fase5.json.gz difere do manifesto")
if sha256(EXP / "formato_fase5.py") != MANIFESTO["formato_fase5_sha256"]:
    raise SystemExit("ERRO: formato_fase5.py difere do manifesto")
for p in (str(REPO), str(EXP)):
    if p not in sys.path:
        sys.path.insert(0, p)
import linguagem_profunda as lp
import formato_fase5 as ff
DADOS = json.loads(gzip.open(EXP / "dados_fase5.json.gz").read().decode("utf-8"))
print("commit:", COMMIT, "| casos:", {k: len(v) for k, v in DADOS.items()}, flush=True)


In [ ]:
# Célula 2 — carregar os pesos da fase 3, tokenizar e preparar o replay
LOCAL = Path("/content/fase5_base")
if LOCAL.exists():
    shutil.rmtree(LOCAL)
shutil.copytree(ORIGEM, LOCAL)
SHA_BASE = sha256(LOCAL / "pesos.pt")
modelo, tokenizer, estado_base = lp.carregar(LOCAL, dispositivo="cpu")
cfg = modelo.config
modelo = modelo.to(dispositivo)
ID = {s: tokenizer.token_to_id(s) for s in lp.ESPECIAIS}
PROIBIDOS = [ID[s] for s in lp.ESPECIAIS if s != "<fim>"]

def preparar(casos):
    out, fora = [], 0
    for c in casos:
        try:
            prompt, resp = ff.montar_ids(lp, tokenizer, c["contexto"], c["historico"], c["mensagem"], c["resposta"], cfg.contexto)
        except ValueError:
            fora += 1
            continue
        out.append({**c, "prompt_ids": prompt, "resp_ids": resp})
    return out, fora

(TREINO, f1), (DEV, f2), (BLIND, f3) = (preparar(DADOS[k]) for k in ("treino", "dev", "blind"))
def estratificar(casos, n, semente):
    por = defaultdict(list)
    for c in casos:
        por[c["situacao"]].append(c)
    rr = random.Random(semente)
    saida = []
    for s in sorted(por):
        rr.shuffle(por[s])
    while len(saida) < n and any(por.values()):
        for s in sorted(por):
            if por[s] and len(saida) < n:
                saida.append(por[s].pop())
    return saida
DEV_MONITOR = estratificar(DEV, 400, SEED + 3)
DADOS_LM = Path("/content/dados-17m")
DADOS_LM.mkdir(exist_ok=True)
for n in ("treino.bin", "validacao.bin"):
    if not (DADOS_LM / n).is_file():
        shutil.copyfile(CACHE / n, DADOS_LM / n)
LM_TREINO = np.memmap(DADOS_LM / "treino.bin", dtype=np.uint16, mode="r")
LM_VALID = np.memmap(DADOS_LM / "validacao.bin", dtype=np.uint16, mode="r")
print(f"base (fase 3) sha256 {SHA_BASE} | {sum(p.numel() for p in modelo.parameters()):,} parâmetros", flush=True)
print("treino", len(TREINO), "| dev", len(DEV), f"(monitor {len(DEV_MONITOR)})", "| blind", len(BLIND),
      "| fora do contexto:", f1 + f2 + f3, "| tokens de resposta no treino:", sum(len(c["resp_ids"]) for c in TREINO), flush=True)


In [ ]:
# Célula 3 — geração, avaliação e perda de linguagem
@torch.no_grad()
def gerar(prompt_ids, max_novos=110):
    modelo.eval()
    ids, saida = list(prompt_ids), []
    for _ in range(min(max_novos, cfg.contexto - len(ids))):
        x = torch.tensor([ids], device=dispositivo)
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            lg = modelo(x)[0][0, -1].float()
        lg[PROIBIDOS] = -1e9
        nxt = int(lg.argmax())
        if nxt == ID["<fim>"]:
            return saida, True
        saida.append(nxt)
        ids.append(nxt)
    return saida, False

def avaliar(casos, rotulo):
    linhas, t0 = [], time.time()
    for i, c in enumerate(casos, 1):
        g, fim = gerar(c["prompt_ids"])
        texto = tokenizer.decode(g).strip()
        av = ff.avaliar_saida(c, texto, fim)
        extra = c["exigir"].get("contem_texto")
        if extra:
            av["contem_texto"] = extra.casefold() in texto.casefold()
            av["tudo_ok"] = av["tudo_ok"] and av["contem_texto"]
        linhas.append({"situacao": c["situacao"], "mensagem": c["mensagem"], "alvo": c["resposta"], "saida": texto, **av})
        if i == 1 or i % 250 == 0 or i == len(casos):
            print(f"  {rotulo}: {i}/{len(casos)} ({time.time()-t0:.0f}s)", flush=True)
    return linhas

@torch.no_grad()
def perda_lm(lotes=40, tamanho=16):
    modelo.eval()
    g = np.random.default_rng(SEED)
    perdas = []
    for _ in range(lotes):
        ini = g.integers(0, len(LM_VALID) - cfg.contexto - 1, size=tamanho)
        x = torch.from_numpy(np.stack([LM_VALID[i:i + cfg.contexto] for i in ini]).astype(np.int64)).to(dispositivo)
        y = torch.from_numpy(np.stack([LM_VALID[i + 1:i + 1 + cfg.contexto] for i in ini]).astype(np.int64)).to(dispositivo)
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            perdas.append(modelo(x, y)[1].item())
    return float(np.mean(perdas))

def mostrar(res, rotulo):
    f = lambda v: "  —  " if v is None else f"{100*v:5.1f}%"
    chaves = ("tudo_ok", "terminou", "usa_fato", "nome_exato", "marca_nao_sei", "marca_urgente", "sem_numero_inventado", "sem_nome_inventado")
    print(f"\n--- {rotulo}: " + " | ".join(f"{k} {f(res.get(k))}" for k in chaves), flush=True)
    piores = sorted(res["por_situacao"].items(), key=lambda kv: kv[1]["tudo_ok"] or 0)[:8]
    print("    piores situações: " + ", ".join(f"{s} {f(r['tudo_ok'])}" for s, r in piores), flush=True)

BLIND_BASE = estratificar(BLIND, 600, SEED + 4)
print("\n=== BASELINE: pesos da fase 3, sem ajuste, numa amostra do blind ===", flush=True)
PERDA_LM_BASE = perda_lm()
BASE_LINHAS = avaliar(BLIND_BASE, "base/blind")
BASE_RES = ff.resumir(BASE_LINHAS)
mostrar(BASE_RES, "fase 3 no blind da fase 5")
print(f"perda de linguagem (validação do pré-treino): {PERDA_LM_BASE:.4f}", flush=True)


In [ ]:
# Célula 4 — ajuste fino: conversar + replay do pré-treino
PASSOS = 5000            # ~2 épocas dos casos de treino
LOTE_SFT = 32
LOTE_REPLAY = 8
PESO_REPLAY = 0.3
LR = 1e-4
AQUECIMENTO = 200
AVALIAR_A_CADA = 500
PACIENCIA = 4
LIMITE_PRESERVACAO = 1.10
MAX_HORAS = 2.5

def lote_sft(casos, rr):
    esc = rr.sample(casos, LOTE_SFT)
    seqs = [c["prompt_ids"] + c["resp_ids"] for c in esc]
    T = max(map(len, seqs))
    x = torch.full((len(seqs), T), ID["<pad>"], dtype=torch.long)
    y = torch.full((len(seqs), T), -100, dtype=torch.long)
    for i, (c, s) in enumerate(zip(esc, seqs)):
        x[i, :len(s)] = torch.tensor(s)
        P = len(c["prompt_ids"])
        y[i, P - 1:len(s) - 1] = torch.tensor(c["resp_ids"])
    return x.to(dispositivo), y.to(dispositivo)

def lote_replay(rng):
    ini = rng.integers(0, len(LM_TREINO) - cfg.contexto - 1, size=LOTE_REPLAY)
    x = np.stack([LM_TREINO[i:i + cfg.contexto] for i in ini]).astype(np.int64)
    y = np.stack([LM_TREINO[i + 1:i + 1 + cfg.contexto] for i in ini]).astype(np.int64)
    return torch.from_numpy(x).to(dispositivo), torch.from_numpy(y).to(dispositivo)

decai = [p for n, p in modelo.named_parameters() if p.dim() >= 2]
sem_decai = [p for n, p in modelo.named_parameters() if p.dim() < 2]
otim = torch.optim.AdamW([{"params": decai, "weight_decay": 0.1}, {"params": sem_decai, "weight_decay": 0.0}],
                         lr=LR, betas=(0.9, 0.95))
escala = torch.amp.GradScaler("cuda")

def taxa(p):
    if p < AQUECIMENTO:
        return LR * (p + 1) / AQUECIMENTO
    prog = (p - AQUECIMENTO) / max(1, PASSOS - AQUECIMENTO)
    return LR * (0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * prog)))

rr, rng = random.Random(SEED + 1), np.random.default_rng(SEED + 2)
historico, melhor, sem_melhora, t0 = [], None, 0, time.time()
for passo in range(1, PASSOS + 1):
    modelo.train()
    for g in otim.param_groups:
        g["lr"] = taxa(passo)
    xs, ys = lote_sft(TREINO, rr)
    xr, yr = lote_replay(rng)
    with torch.autocast(device_type="cuda", dtype=torch.float16):
        perda_sft = modelo(xs, ys)[1]
        perda_rep = modelo(xr, yr)[1]
        perda = perda_sft + PESO_REPLAY * perda_rep
    otim.zero_grad(set_to_none=True)
    escala.scale(perda).backward()
    escala.unscale_(otim)
    torch.nn.utils.clip_grad_norm_(modelo.parameters(), 1.0)
    escala.step(otim)
    escala.update()
    if passo % 100 == 0 or passo == 1:
        print(f"passo {passo:4d}/{PASSOS} conversa {perda_sft.item():.3f} replay {perda_rep.item():.3f} lr {taxa(passo):.2e} "
              f"({time.time()-t0:.0f}s)", flush=True)
    fim_tempo = time.time() - t0 > MAX_HORAS * 3600
    if passo % AVALIAR_A_CADA == 0 or passo == PASSOS or fim_tempo:
        res = ff.resumir(avaliar(DEV_MONITOR, f"dev@{passo}"))
        plm = perda_lm()
        preserva = plm <= PERDA_LM_BASE * LIMITE_PRESERVACAO
        historico.append({"passo": passo, "dev": res, "perda_lm": plm, "preserva": preserva})
        mostrar(res, f"DEV passo {passo}")
        print(f"  perda LM {plm:.4f} (base {PERDA_LM_BASE:.4f}) | preserva={preserva}", flush=True)
        if preserva and (melhor is None or res["tudo_ok"] > melhor["tudo_ok"] + 1e-4):
            melhor = {"passo": passo, "tudo_ok": res["tudo_ok"],
                      "estado": {k: v.detach().cpu().clone() for k, v in modelo.state_dict().items()}}
            sem_melhora = 0
        else:
            sem_melhora += 1
        if sem_melhora >= PACIENCIA or fim_tempo:
            print("parando: sem melhora no dev ou tempo esgotado", flush=True)
            break
if melhor is None:
    raise SystemExit("ERRO: nenhum ponto preservou a perda de linguagem; nada é salvo.")
modelo.load_state_dict(melhor["estado"])
print(f"melhor passo pelo dev: {melhor['passo']} (tudo ok {100*melhor['tudo_ok']:.1f}%)", flush=True)


In [ ]:
# Célula 5 — blind (assuntos, nomes e formas de fala nunca vistos), gates e salvamento
FINAL_LINHAS = avaliar(BLIND, "final/blind")
FINAL = ff.resumir(FINAL_LINHAS)
PERDA_LM_FINAL = perda_lm()
mostrar(BASE_RES, "fase 3 (sem ajuste) numa amostra do blind")
mostrar(FINAL, "fase 5 no blind inteiro")
print(f"perda de linguagem: base {PERDA_LM_BASE:.4f} → fase 5 {PERDA_LM_FINAL:.4f}", flush=True)
v = lambda k: FINAL.get(k) if FINAL.get(k) is not None else 0.0
GATES = {
    "tudo_ok>=0.85": v("tudo_ok") >= 0.85,
    "usa_fato>=0.90": v("usa_fato") >= 0.90,
    "nome_exato>=0.90": v("nome_exato") >= 0.90,
    "nao_sei>=0.90": v("marca_nao_sei") >= 0.90,
    "urgente_192>=0.95": v("marca_urgente") >= 0.95,
    "terminou>=0.95": v("terminou") >= 0.95,
    "sem_numero_inventado>=0.97": v("sem_numero_inventado") >= 0.97,
    "sem_nome_inventado>=0.95": v("sem_nome_inventado") >= 0.95,
    "cada_situacao>=0.60": all((r["tudo_ok"] or 0) >= 0.60 for r in FINAL["por_situacao"].values()),
    "perda_lm<=110%_base": PERDA_LM_FINAL <= PERDA_LM_BASE * LIMITE_PRESERVACAO,
}
GATE_PASS = all(GATES.values())
print(json.dumps(GATES, ensure_ascii=False, indent=1), flush=True)
print("\nExemplos do blind:", flush=True)
for l in random.Random(5).sample(FINAL_LINHAS, 20):
    print(f"[{l['situacao']}] {l['mensagem']}\n   → {l['saida']}\n   ok={l['tudo_ok']}", flush=True)

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
DEST = SAIDA / f"fase5_{stamp}"
DEST.mkdir(parents=True, exist_ok=False)
estado_cpu = {k: v.detach().cpu() for k, v in modelo.state_dict().items()}
torch.save({"versao": lp.VERSAO, "modelo": estado_cpu, "config": estado_base["config"], "passo": melhor["passo"],
            "execucao": {"tokenizer_sha256": sha256(LOCAL / "tokenizer.json"), "pesos_externos": False,
                         "base_sha256": SHA_BASE, "protocolo": "fase5_conversa", "commit": COMMIT,
                         "dados_sha256": MANIFESTO["dados_sha256"]}},
           DEST / "pesos.pt")
shutil.copyfile(LOCAL / "tokenizer.json", DEST / "tokenizer.json")
resultado = {
    "protocolo": "fase5_conversa", "criado_em": datetime.now().isoformat(), "commit": COMMIT,
    "base_sha256": SHA_BASE, "pesos_sha256": sha256(DEST / "pesos.pt"), "dados_sha256": MANIFESTO["dados_sha256"],
    "hiperparametros": {"passos": PASSOS, "lote_sft": LOTE_SFT, "lote_replay": LOTE_REPLAY, "peso_replay": PESO_REPLAY,
                        "lr": LR, "aquecimento": AQUECIMENTO}, "melhor_passo_dev": melhor["passo"],
    "historico": historico, "baseline_blind_amostra": BASE_RES, "final_blind": FINAL,
    "perda_lm": {"base": PERDA_LM_BASE, "final": PERDA_LM_FINAL}, "gates": GATES, "gate_pass": GATE_PASS,
    "porta_conversas_reais": "pendente", "ativo_no_chat": False, "promovido": False,
    "linhas_blind_final": FINAL_LINHAS,
}
(DEST / "resultado_fase5.json").write_text(json.dumps(resultado, ensure_ascii=False, indent=1), encoding="utf-8")
for p in (DEST / "pesos.pt", DEST / "tokenizer.json", DEST / "resultado_fase5.json"):
    if not p.is_file() or p.stat().st_size == 0:
        raise SystemExit(f"ERRO de persistência: {p}")
print(f"\nSALVO EM: {DEST}\npesos sha256 {resultado['pesos_sha256']}\ngate_pass={GATE_PASS} | porta com conversas reais pendente | ativo_no_chat=false", flush=True)
print("\nMande um print deste resumo.", flush=True)
